# Daily Challenge: Build a Retrieval Augmented Generation (RAG) System

We will build a full RAG pipeline using **LangChain**, **FAISS**, and **Hugging Face** models.
The system loads a real dataset, indexes it into a vector store, and answers questions
by retrieving relevant context before generating a response.

## Hugging Face — Quick Introduction

Hugging Face Transformers provides easy access to pre-trained NLP models (BERT, GPT, T5, …).
A tokenizer converts raw text into token IDs; the model maps those IDs to contextual embeddings.

```python
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model     = AutoModel.from_pretrained("bert-base-uncased")
inputs  = tokenizer("Hello Hugging Face!", return_tensors="pt")
outputs = model(**inputs)
# outputs.last_hidden_state → token embeddings
```

## Step 1 · Install required libraries

In [ ]:
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -Uq datasets
!pip install -q faiss-cpu
!pip install -U langchain-community

## Step 2 · Load the dataset

We use `HuggingFaceDatasetLoader` to pull **databricks/databricks-dolly-15k** from the Hub
and expose its `context` column as LangChain `Document` objects.

In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name       = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data   = loader.load()

print(f"Total documents loaded: {len(data)}")
print(data[:2])   # preview the first 2 entries

## Step 3 · Split the documents

Language models have a limited context window. We split large documents into overlapping
chunks so no important information is lost between boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = text_splitter.split_documents(data)

print(f"Number of chunks after splitting: {len(docs)}")
print(docs[0])   # preview the first chunk

## Step 4 · Embed the text

`all-MiniLM-L6-v2` is a compact sentence-transformer that produces 384-dimensional vectors
capturing the semantic meaning of each chunk.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

modelPath      = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs   = {'device': 'cpu'}
encode_kwargs  = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Optional: test with a single sentence
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(f"Embedding dimension : {len(query_result)}")
print(f"First 3 values      : {query_result[:3]}")

## Step 5 · Create the FAISS vector store

FAISS indexes all chunk embeddings so we can perform fast approximate nearest-neighbour
searches at query time.

In [ ]:
from langchain_community.vectorstores import FAISS

# This step embeds every chunk and indexes the vectors — may take a few minutes
db = FAISS.from_documents(docs, embeddings)

print("FAISS index built successfully.")
print(f"Index contains {db.index.ntotal} vectors.")

## Step 6 · Prepare the LLM (question-answering model)

We use **Intel/dynamic_tinybert**, a lightweight extractive QA model. It takes a (question, context)
pair and returns the answer span inside the context. We wrap it in a `HuggingFacePipeline` so
LangChain can call it transparently.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"

# Load tokenizer with truncation enabled (required for QA models)
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding=True,
    truncation=True,
    max_length=512
)

# Build the Hugging Face QA pipeline
qa_pipeline = pipeline(
    "question-answering",
    model=model_name,
    tokenizer=tokenizer,
    return_tensors='pt'
)

# Wrap it in a LangChain-compatible LLM interface
llm = HuggingFacePipeline(
    pipeline=qa_pipeline,
    model_kwargs={"temperature": 0.7, "max_length": 512},
)

print("LLM pipeline ready.")

## Step 7 · Build the Retrieval QA chain

The chain links the **retriever** (FAISS similarity search) with the **LLM**.
For each query it will:
1. Retrieve the `k` most relevant chunks.
2. Pass those chunks as context to the QA model.
3. Return the generated answer.

In [ ]:
from langchain.chains import RetrievalQA

# Create a retriever that returns the 4 most relevant documents
retriever = db.as_retriever(search_kwargs={"k": 4})

# Build the full RAG chain
# chain_type="refine" iteratively refines the answer over each retrieved document
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=False
)

print("RetrievalQA chain ready.")

## Step 8 · Test the RAG system

In [ ]:
question = "What is cheesemaking?"

result = qa.run({"query": question})

print(f"Question : {question}")
print(f"Answer   : {result}")

## Bonus · Ask more questions

In [ ]:
questions = [
    "Who invented the telephone?",
    "What is the capital of France?",
    "How does photosynthesis work?",
]

for q in questions:
    ans = qa.run({"query": q})
    print(f"Q: {q}")
    print(f"A: {ans}\n")